# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNq1PWtz3DaS3+dX4JhKhYxHzIxkO85cJlWOJSeuk22d4yR7q1WxMENwhhZfIciRxi7f39rv+8uuuwGS4Gske32ujYYEgQbQ6DcaWMuyXqZFmi9YAD+2dNjRTyzmUcTZySmzfy2TfcmTk9OjY/YdO4539L0Qt0WZc3z8ORKJL3J3MvldpswXLBfrMpepXEwY+5a9Tf1Usgj+i1Nf4O9fpfB5wtY833D8JhIGff1y8Tvj+LAT75mdiTwII/by6d9evHzNvgNIjD09f/vamTIuQ5akbMv3TEioxxFCkQtWJpymwPYIRoabMoQPwqVhnENHmYBRMSlYBkPE3hPd95unL5m95uutwPFnfBMmABUez8OkvHVYJPYwx5RGgTPg+Xob7tTQM57zCOa1YBpoDgPRc2UFz30YitiU0FyqkZxFrMj5ir+DzyLOUmjOdhxBfeeL3XdyGzPbD+U6xSIYmaMnoFBEo08LsRFsnSbY1ZonPuDxP6uFYPYzqPeAmkKdHDAjEhp6NVRqaaKIbUQCiFyHadKsRoO3XMgyKmitoPtNiZOiiohxhTY1VAazgYWRW8RdVK1Gg7Z3pSyERExloghzrBKHMtYVhWRhIguewP9E6k4sy5qEiKACBww0lcNY3aAEwoOqsEDroPq+WVdP2HsUrqrXMK2e3sk0qZ5TWT3lonqS27IIo/qtXAGa10LWNYttLrgfJpu6IIzrxu/DDMhVTKr3pIyzPQ4xySZBnsbs4sU5099exHwjpurndSYnqoLLmqkG4WYy+fXszRlbwlDdjBdb1w/zhMfCrt75SuKv7XnYsec5zgTWzDt//ey/oFU9WPc8XV/bzuSPF2d//gYfbAt6SwpryqwVX1/jbyQCes/DzbawnMlkso64lOwsATYQC1o9XwTM88IkLDzPliIKpkiFwfJVmsBUonSzzPIwKRxVG/9hJTeDHrEeA7JQ83J9UYh1ASNqVQQIUBX+totzEa/wA3bT/iK3PBPjX7x4d+BjcbDhyOeMwwTHv3jXYeLDZ6DZ1sd36UpC8QeLrwsUGdaCzQDZIvHWacTx7WPd4CugKZQIICqFHyK72fgATJLuwsgh/gUuAcIDIbMGoQxME3DgzHaXouDUY5DmMfbwxMUetcSG95NHVLBS4gIKHs9cYxQEwwMgHhKYQYLv0jCx9TK+/v2td/riDcJVQ3aRv6xmWYt83xCDOTS3zHxeCBvrw7pz304zkdjtXh2ngSRu1yIr2Bn9gIRqg82AVNtDVwIJEPCxS0zc30P5cx7JzgpqVHhZmkZQYx24b4mBLuD97BbUGahHO+a33k2aX4tcLo9BMqq1v7R0mXXVDBnQEfNrWLpc9hAmbkNZeOn18m1eCuA2tfBH/8Y/pUlrPgXlhmj1bgQytNTsCiI3lUsbyCBZgxj9zlDrFlFHrzjeWY7B0CCNz4VgRa3OayWIKpM0K9GnL97xPKrVOmgF1zKYgoTdttyAbNkEfC28bbmqRJ8EVebR8nmAuqYJCDjkocurceoCWmc5qI8ODNtxaeLtyvgvDFhO37zQx3Yj1WrYYke1oMkulECFI3Wb8T6AAcsitwOXBDRRNYEKVHc7KpdXo3SO6kMM8BBISdt6lZIlUPopGCeCNGltv/z6fAFLKpxW01wA9ydsNqlLkVZy36aBtbtJANmzVslNCJKA+BSro7JYQRerMggEyP3NcubgaIM+Tm62MEmGpD6MrxV0pVjTfvyQ/fgjO545gxVhvZK0YKtxtOsZJoMVElwP4HJ75TQYQHGOWtzFP4ZKotkOC4H5Y5qpuG2Po0gLjpJDlrEtboH5Mzv3p4oSnL6uswPrQnGMthm1GQzMsmAfFDCwtb/99mS2cOfBR/bLz/jZ6k8tsD4YM2BHrFi4s+CjNCSxxgtBnTTaXKm7LMyEISGmaPhMGTRYpQZRKKbdn/hgJyo9CQ8V19Yi4zR8+zxKb17yAgRDsrkA0FFoKEvsC3B0V30Xe/MAM2Alw6tvd+hYjzJIIxDZSxpvKYUneQCmRSJTEM9KtBqkQxNqL9lX7DmYOds/TgHlvlinfhiEa3JAlO/zv/PZLTyC9cuzUNv+o+Knmp4rEg7KxAsQ9M6P7Xjt8WiTLq14bbUJW7P7230mzvIc3K8ewMFeRnvqs809BEpfsLAaLehkgR+SpUkIHQ2IlHvC/2TYmmBxlg29kqWgCPUGDB+w02iV9RuZYGYBLbhZ8FfJo7DYk8Xa1mrPkAFNB9Vlb7E1sx+hdQHlD+pxO+ikiAhrpuTswNDAnDVIi7wesLuYBFcLmbZ2/tgzQBG6cJscPDbQpDxegfel3OHKGwbgNSjtQyYILoLRzNRwKjtQj+IZzAv8v5aSrTyUFFR0I+3wzUWrH4wd6a5Ln4OgKuIycoHg0xuvCE6OgT8RaQcaJcmnVl+BbbEFk/W6W7slVpStr0f+M0DY5Ckg7o0A41fkJjOb68lCSRZ5m+xaNZa1pQbj9tLE22SldYUuCSqUhlru1gvQt+GWDPbcclt6s2jDarpmQBBtB2UcOHES0SxQFTOjM5pqFfmR1OKO67oduTMoVTrOkTLGDQ0xbDaOquKxf9ZWN/eP/LA42h0fzY4IASAA2jL7M+WLQsFdgmuARFp02aWx9up81tqoVZkCmxO3DqxLy6295wIMIRRKycPpkxr4xDiTcbIj3/czZgeANe3teB7CKoGJXHD5GcSnve97TR/8k0+nwEEijHdflgobhByixM6y2C2SA+k0sjBOf9U+c83aq3WINu9etv/fRRtasj6d/xur1VqnKbpWpUQlK1NQwHMKT46sngoL1UuiXgfXo6XtCnE7ZEJfYPOe4UwjLimOZF8auKDOTCl61VErAvADbrMDFsxwO+uq50VL9McT6u8A3oIhoqLRM/vDN9T7NziY+pEgqgF9oyyabz4OUNpBu3f0Qy9IN4zQw55Fj+iGRW7L9Rj2Ve827Q0qstTSaQwtDg4Jhi3FF8XA4CSdyWHwVawTxjvs1IPbfP2l3JH7KPPDQVrTdKv5E22/Po828U8pgIkoZAquZfheOcloREKhGerrdXBppUGA3op1tbhjoJUPR4a+t85KTzftOHPINXfYPXo77sXTNr6YvQYJlmMQfg9+CjoYzc6MY7XEGWJkyFLuB9oyJFa7Jf+nXXVlFjiD4bfMXAaSoHWJaY6OuMBukdoWoMxyhqO8LYOOvtzwPC4zezAa81JvTUagClUo5s6wStXaiwU637WjqrvBz20/8/eEowPHfdz0SkKw0cGrUzuGC7ZO4yyMlBt6LfIElhxX6tnvp0/ZHggg3DSzoeBrtEnzsIhV+HVdnr56pXZGAUaWw6jzaveN4+Yix7gtbq1G6fp+3qIO1rQcg5ZRYpIOVdZKr3KslCtIXmYoPb7jMD2gSLtDDMrZv9P16gd8h3bVTnN+02b7GODRJzcRN7b15pefn4IIsR/Nj6cM/jjwPJsy/T+nzXZ+1RbBuvjHDuNOFVdEUZhJYdvz7x9P2UMAc3ICD8ezmUPxv2hpw/OUzb/HP4/gz/GjR92OgGqBOZNNhHB+gObzHxDQHCF+3wB6DIVPEMzjQTDociqs/0LbuRiv9MUuXIulhetgOeDvJ2BggnAD0d8JtFLQUzUPk0DkqBdIMtnQvyrnZZGuuSxsBW7K/AJU3FJ9RMFVzB87w4ExwBwgchnGU9wb9ZoeZCEyuYTVSNdFLoSXC7C5StQPy/nxkxFbEUGswZ65lstKYqlX62qq97JhUMsNQC2LrARJjuO0CmQMue0atw2hyn2y3uYgOt8L25mMGD4YwgFTGZQmsvJ9RMUnRfZZBTzGrfmUpXFYhH5lfhrx2yEFBVIXHo016Om9vs7rcuAgp7XtVtcPA1rAIiQFltP2mchdRK9+qxjzJRS9oZLJiBbUO2TmZFDywM8dLVRHdQNNCaq0Y9kajfXwlsbIgE0CXkaFSX1dcJ/ktphzWdKS9EeDq9ed5+fuAa0xlshzDAoC4fCE1Lze8e1STqWzOhqqt+56/TJJeREDX9oKA//tUNyqBu4uzAuUNdBVmu/tMVGlwpBUx8O44poXIJucavuDgvdjmkS5E7MxRsWElJL07Ydd7JZS+K1Nle+wtLPXMlN7Lf/659Bei9pvQfb/sNEQNJF8s8t57G1W31xhc+uT9q6/xAYwCddESHMPONMCIYw3uLeD8UdvtVFbIy35ABUOCYa2LY31pcrSWaNQB9iyyJ0u5WxqtUubhlBgmJrN19eZdMVtGHjgjiRgs0rRrrrlsN5RtsV8BvjgojpiyyVTepxsxiRzeZ7zPTW8BIcS9OaVCwYWkMePoCRn5tBrNKimadH0MDSDJpiL0FHq7ERekBVhOd0JdWs8NcjgK8wNA5YBDgVqS1fvBGWpoeOVI9eqPK6S+zla6JidFYNdKBqhjAgYnGldYy+n7FaqakmavBd5anP2E5vPWkY+boHeys563YI5cTufsj387ucA41YqDE7pid/i074u2+uyAYwREvI0s22EuSe47AGbE1z4NbB2M2Vb3aIlAnEOIKYx3cLGOg77ls3d+RPD8v5r2KgDDMhRg07+BeIfjA1Fs7YtQVPfgJgB1sem8LZVb05vIwqagl5AFWvPZ8cPwf6Cv47O4HLPn7569vfXvzmkagnhxw+VYJJ/NUJXZlFYeHIrRFEpanwGAyXHUILKyppiKtYUd0+mOhmr5UCchrvQV2mO2/QdZlro0J1KQwSfQWCCHu4O4X4ybk8FaeKnbWM/UD0fZHkjdSbfrHibFcY4ty2GVR/KqzQkhp41ygyFJENK0DfHGeWzFhPguGouILTX9daUxMNdnuxNI1eKDVAIjASQiCPgeUFJJTqOODUUCTq54ZRRyocAYxPtSYGudFfQBWxXixIFecDV7fSoNiXDjoevUaXAHQbVy2EyZ+jyDHDpA4Fjd9CP0+L/IchDzVBO4HyN1n8SUsn3u5xftdCKeJQqekhxLCxCQoN6wFjycoYr9Ccw8sydPbzqyiOsTsJ6QaISiIVtS5CXcoE7nQWmhW7AkhCyl6SCrYmBXOIv25pajtObG47ORpESwgj+BGWfOFMSMVCAMqkuVfk5IeXngDcm7MRpBovZlwQqdYF6w8xWtVOs3R1D0wr8j3YqGrZBUCB0lNh1EMB7gEcdTGnEHTIDIJf49aq2clG9IwcYwlbLcCpFYQW4d5xejFwzOED8IoYH7Wo1PgkMi4e5FnDoC8Bffuthnpe8bzqJfSq0ByeeQzu9c4oTwRe/XMPLPU3x5+iSilyDaJCAI0Mman22HRvLe7UGh9OrPGo977Ee2KutjzESUv3JRUfkN1F0NGoMTOn7Hlay23WpbxdlJPj2UmHaVUjugcBvYbLxtOnjl0CkaGN7VXP7QBNcTQ+sCQ9cihCj3p6ANTvQZB2BJvC2KTArKnB6QBfoeDabHejHS3N0dhWheOt0S75l1PUa0LuIMauAMsEVZpxBz0Q5+u3Gajn1J/et+rVjwqO4hSnCJ7J0YsKlUaAzwZcdXvpEt75ew+HdpmpkAyHuwX2PA1OtpgsYBitybKUVZIUNtdRuEN56anNGduh7tEEUVcvdqz+In+PDgeyczBc6iKACH8p9Pe4ZsQbNs58aKbMYWnNDdGjGbeQSaPdlI6O6cpJQ2xhxFAdF6aVF3C4UNyiyMYZVRbBwPYU/ZZsSBg8mj86m6zleQz50XEZFiPriegGWAKqKa22FUE9uWIgYllHZHUb0vFEv1Y6y0o0ED/Ezb4wjsC/URuKBgD0UXNVpc3oX8XC4/8ph/+iaRtVY9C5kZwiUH2tEl02QBzpv9VxcdePVibgt7KzZpVB9dXcbmnXGzEFClwph37mtYdChpFSFoXbFoWYBteyICSQgFG7gYKFhQu8geJ44ymarStiPSzB0FU4efd7eTVpsRd4ZcBunncH35zK4m6PhdnZ0WqWIquHdHKo2sKNTJzvSJwphj4VuRZwVe5X4bXddc8VWMEi92oQ/4qhLfRymoaNPDKLT2sCD08ke/uKBdC3KWrH0zXAw3aCgUUOpH23HVqrUmR4MsI/CrCSeJzGCvaR52FWh0w7Mjw9sMGAPHsSQaJ7qFf0itqwOl9bSXsdwW9asFvhoL3xuoF1DantwYVDDZP+xBEwVvCgqQ9rCSDEQSSd19cBedT0+kxHIeWlT/RUmhqii6yulb0g0jJ0NI8VDmghGaF87Bw5kfMX+UNlFuVDUic92kNNxwwd4cpNHwLau6zC1awlWCp6rYFHKuA+qX3RdoNr54RQeIizWnCAruwCfSeMpLsdXIJ7PNtqqc64Yk2ulS6ERR+ME8p6qPCE1uQH77c7Bt0bYuGgqHe3LEHeqU9iRsq/Fvm3ApFlh+mcoaPQ5SpT7c0Og1jRCBwrbeEM1alJHP1RyMO9iW50Mu3aBXEheOuOVdqATVvsCrdrLxYL98L2BwW3rjJlfxpm0cYq4TnnhwfT1EYGBjjT2t+5W3PohODsFdjA/vmrMwLxMNPpU1I4ht1RHIYFh9CPxjH6m0JZ67ofJvIMxwG7wftq3HHQnSnrrF7Jn9TMQcTUOUFXL+fHJwzaUSkgv6WQg9Lu0tiVuI4W+gAFs6STZcu4+gY8piCaxnEHF9uJD1XRpPeNgw/DcmqoCVDBV1zzOomZIgOQCd3EG5qRFmc7ez0UUih32+Ng0pckr2+TolkU8XvmcoZBeMI2KxMPQUcijbsK/aSxRrsgg77dScC5q5p10DbZa1WLWw8WlRc+tk4C0JrqmflFV1YtZl5ZMV1XPqiY9mxXVxiFWwydVqZOWVGNfV2zeVfX63Wykl6ial37TM1NvrWHMOskajTGul8YGMjkG9fHfZVhQriDFoy1zmCAu1GFV0kgLI76lmaumfqc6s2xWIkYx6/QNC6XEzEbEYm3AitfMSlRi1vo4GEMflmem66bMZzP+r2PgDe87lWc3aFqbMnXYsmihE1U5COJJF0Kt9UesE8wtBKUdlSpp0bae49Hi+hi/L7SCM5YPhSr05YdrzQbLliu8rDxiJYzor5JG8J8WRrUBPWwS1pKpcadRPMF/tVhSP7VkUj/TEfygkKK/poCqnxoppX+HwTTCa5WmkV2/OtNadqkP6sVphJiyifWbMwodZFwFeZUayum6kkSowU3dbS72dU106kQo+WH6CHcofR2fVR+BVi4tKAK2HrOB3lSXP+AuEzVCC+d68HRpC3AjDvCEdPcc+YU6OO1R5y145uFprNI7Mt0KpQ/FRxafaHnsXMl3zdUKND7Vc2CRted9SD66WbLB3YW69Vm1GMI4gP8Ve4MRLOKXCHz1GftanQBDu/avUtCNIJivgQqLS1FdeMKBsfIwzY2TVGgznl3qU/xoqdtnl/Up/ivtMiB96TQIzPKGGtWxfmzxsMGVVx0twS0Y+DRzf5jhZSRNB99hn0YDpWWqhnWLO0eh4LSOGRfh+toGM3mFTL/GiLWkxIE066yVZVlPdzx5T2JnhTt9TJawNjG5DT7ul5MkoqteFPryDW9tcBLy5iNphM2RZDICYACYUiIx9t4ni1sdkYGJP5oyu5PlNceZ4ia1mtF8wFit1SDH1VtBK46bTbc0/351Gs4NDwvo8ZHTxmAG9rjoozBIunwLMFr3f5ztAGkdDNCtDk0VdczZLjDloFjiWk0ZPMvlyJJNmc9BKybL/mEZ4B7aQLTvEcDWMiNIOpUDMCCjaOjACeJH9vZJdL/EtQbW6ts3LuvLNq4w6Dg3ZWV1ZQqIuvU15h11EnQMO+YErISzBDeX+aK6rwh4Fs1dliJZquOaVicQVPUwIF67gzsyB9epVd0e0pnCKHKTnT6N3hJ9JDINeblDbS+H/CzS62McVMWjKPzi6/gpESjgCVhFy4ypIV3uu2Vnqagb2or1SdqTUwv5x+oeL7PImN81zr41cnKh/4/A/euf6pSW1Yk2A5j7Dle5HgszRErbAwc2Bj7nyNqAQVQd0h9YulrUdySWHBdQ9YpZ52GchYR/zE/K1VLwTflO5Km0humgPvyl938VbXR2VFobU+IWdz+GNK7F83UYRdzdRCvrcwiz0Ugjh4jURx+W5VrrOEW7pqabzadKAxIN1+ruExfPugiToqJlM5L04umnnoJrEVoVmezGJAfQNXx0ZAQPQ5HJhqpUnTtpqoJ7aI2VqNQxy/A9GJejyz125Gs0B/XeewP3y9EcvGfosJoaktot2W4YXWiPPWrbYQ90kebjSZ+wlUlPq9buvmWaGaCbwgY4lE0Ms7W6PC4oBV3iE1X3yjFbK7s91xH89tVxdJLINSCd0r0muGOO51S0gbsAkw3EK5EEr68OmmLCMilPlWUNpKDcMul2MmgO3kFFNCmLPCC6tL7+n6/jr33v61+/fvn1b56FdvGIe0Ex1hHvoliN3QKQZN0B3UHWRtiIb1QSkaVQgbF1mrAypRtXEqMu4OTiZoCVgJEG2LN0xlzd9sqEW9CVV9JboPRSqVgeOYLU5Ufj8ND6euiCuJa1iXffbWxqOq3M1UX/tAjCGjLTYDCX1BiJkExkowgA9nmQ7zbaXqGqjWECNjaliGGpMxnRXSiZQKODnf49yCR6RzsbgGpL26DOc+WGATHTugvMvlinoPLIBZMLpEghi9IPU5aVAnwNvsrDXF+3ActrwBJJdQtbZQhGvKJxTGGYUi6buqQyZ+pwV5e81bWDLgxh3xaVeP8fXhSnxuJuizgCuwTZoe0BB2rHxlSbB0gR8FSUWST6R1+7rq5yr+tQf9fPyNfDTBD0wv5VJcK3tKHlgK9logFqdOdZFirHsHtZmyvLVQwOU99s1rUw0S+bqs0UI2hURYt0FMgIBtUhoE5MumLMOpaD1t1U62XiESxYymJRc49inY4taVeavAlRoxhyaG/NbgrVkfKCst01v2v+bycw1Me2i4bbG8GwUhcrWv9I/pFYapkCz1X3eeoExoDEBGLYMRVUE0Yw9IgZW1CFXa94dSAlrt6Nsc+mbODSPyDbG8v59MMTSigHaTvRMuxJaSICjPTijnnavbCwT6lhNzqhO+ncXAjVDCjZ7nC3HlHafTrfdXvfDfW96xhNNDVwkvWJJhIMpGtBKnz46FQ7Y+GOAsqdr3eBAglBVS+vHBhngQnD5KSBbAMaqmEa1T7Z2wFMcJ1OrQbQA9nZW5WuCtETLlr0UM2V0j9IuC21v1rFabkX8qUypKb1RU0eGWJLurfHLirHaD7iE9atNMS6HcG8s5XmqKpVh5nu0y0eYBpuPqPm/XjyMER9Oe9SbZSq+5eXF5eWeqKDlVmJBZRY1GC64WrEuWbsMFPMjOIXo3Z4wFMkEhMUuFyH4VKn3GvSWuJ5AGNXAdO6+yykcYVhaYSM8N2bPIQVRjHnDO0+/fAEozZxxkHnK0/s7y8uDL57T0ICLMwHzHLfh5nVDt7oa33dv4fZc/i136tpTZsPLy6807Pn50/fnp1O6RA3dhyBeoiWc7op8H1/Y4g0Nu5ECx+niWfOMSpPWqCvGREoDNLs8be3r9+cnSKrBC7es4VjtW3LjbOHqPtJfeOvtgWULgTJ2knRM33EwRn1ar/X+O4vToCHlBsMqPwdfHO6lk4eYzBEWwvhJkmBJgRuOHWv7auXcU7X1Z7j0fzWtYb11hNgb0mjgEkscY2QEpeKHK/FfnntDNwQq7bJciHHTvR34pRYU2di/BoWdO4+wrOUGI6unKejn8DoxOgMGqMuEACezcdQv/aHqP3LF+dnsIKvzvA+5g/NAR+t5Rfs0rboCkJYOwqnWRr8go718To6pioNs7MNhiPQfCQKvNXIxePuDRhidYJCxm5VbwwSDP52X9/ySuCemODWPFqXEcFTNUYBkUuYVyeUENJJe35UQV3CMAol5pm6nF0bYggGz903YKoPo6hRJrpfxWw9jtcx+ISpRzigM1UBZ/TL+c+s+nxveDdipZnPxaP6FgLBKxzA0aW7++D7/YGB5eAGq1sC1hnc85//dg84WSjec8L290gGF/SqrmoHJ+s+06oX/ntc+JewAA053HseUeojiCePAcQfKtlLSIUWTA07f33q3BuYFmmg4Eg2PFVWjMGKlmME67SLPcBZf+CHFl8JuRbJCFcNUvCxAafmJUXDY1D0CXQiK/QTiRFwHm+qKAiOhAaNU2rqHYa3CYG+Th7PaKF64OqvI0DUiCuM0oQaVtTI/NgkQWllXB0y/lwny3CsBjaIjMwf8myWmNxsqfSeKvunTvihbJ5uus86xls8LnXE6Ofzs1enZ3hr+RFpxqOjgK+LNN8f0a5VmVEhRWVm9HjRj9MOOemVS4qsmu3JODk6Uh2EuLqEHngBDFkKT12o8BEaWxp/8KZQaNFRSVu9KLAKp/qDenGGwCmar3FflVDOj2pcvyvAVYaP+qjfBkHXa4ZTnFfbJzpwpbZQFPqOGhFNox3Le8DecX0tvc6NA0snpUgxN2qTXAIdnOp8amJjjf7Ge68614dnubq9Sh/scS/I5ARawf59WJ+l+fHFxRmVg5Vilv/29vT172/viuOjW6AT1lZlQIeh5s3YfNyUW3ZOnuKA0UTMcleNppfPYD2l05osyHks6OowaqPbqi1YZRVaz/ECiAGzMibjB9w79ADtvAWR2f/wHziwGAhtcIclViHohg11uALWhCIWWuAOJ/s3rWzcXqfddfcETHBKjXXxxtTMnlPs7yFdozMgFwd2ooDHizAp21sXtPjV4VZCTq5Pb/bjVFqEqNsYx5W6ie5xbV3VGkB91RF4aUa6tBF33PM7Qo9dSCM3nvWI4RJljNNdu/4I9flnG4znOv7bPgsdS2cxdk97yH5aKsrGjgBGhYvxy+00H9B53PGjAQbhdHMogFcoc8Jg+3RT3BYq+KVDX0QMraBLhhf6o+zCNGAMts9G0qAUtMujk9lstugna6vPk8nEO3v1y4tXZ9WCTCaoK0FeeYL+31Ts7v9VyiZKVzxiutlED6uC0kvUa8CfNfBq51cPpgL2f3De0U4=", "blender_rig.py": "...", "app.py": "eNq9fV1vG8mW2Lt+RaWNAZs22SIpybYoUxuN7bENjO8osu7cIByiUWQXybaa3T39QVEmuNiXXSBAggvsPiywSLC7wCJ53SAPyfO9/8R/IPkJOedUVX+xSUl37mQuLtUf1adOne9TdapsGMYHPxHRlH9ht2LMzHcRd9ygyRzhMREnKdz04cZPooCe+UEixkFwwwI2CRYBC/nM9flBmI49d8LhmY//TyIeC58zzuBqKWL4lKU+S1IfQMD1ay9InanHI2EZhnHgLsIgStjMC8b6+nMc+Po6iPVVxH0nWGR3Ql/FYhKJJGsWp+MwCiYizp4k80jAwPxZ9sBdiIOsYxo04zFcHRxMo2DBLKbewWim7kw9FD6MVuhXM5HY8snBwcVvPnz8xAZsaLzmC6BIZLSY8TqIIkFXn7iXOlxfJvLqW+568upKhEEcGKODy6u3n95eI6D1AYP/ngDBF+NI9JkZ8jiIWyyYJJEQLTbhQOMWS8QqSSPetFi3w6gJcGYhfPg7YIsAWBaw6zQaBwTOoEsDoJ202NHL4xZ8hf+1WK9z/LLZko2ueOg61KoHzXonz1vseLvVBQwD2xx3FKiX220+8pW7oFYn8Oak24MOT2Sz487pc2i2OXj77y8+Xn7/9pP95sMV4GwcovyAwB2+T/27lPtHb9q9Qx7HwN9DseKL0BM2AJ2J2Dg4OHDElNlT1xOx6TT71GsQOSIiVsjx29yH9k5gzbwxEls9BXGvPIncmTUdr/In1tgTvlNoIantfsmgUYfZf7oZKA5P3ElgBePPha+zx4vEKzwGaYpE7Ab+kWPF+KYCNHTFFx5rXEMBPNY3PJq4nsf1LQzAdv1pYKH2gDTh51MUhCF8EvJkbn0OXN90WmzaZNMgYlPm+ope7pTpRmLlxklsbn/TzEA+A5jTHEQM6iAcExXYwp+tb42nRrPZLA4M+puiNcHPASAoNt1NQcec+NZN5qZpWF4wM1q6W9DwNMLGmu1xuljw6M7EISvecxgsUQB00zT45zROQE5abL2RnXugrESQDJWp8fTpOvukQWrVaLHGXzSa/dZGqtnTp+wP/0urGltz2RTuG80NC1f40igAFPHPqfBEErACYOANtjYLj+Yp8hI668ALedNixF8G9qUEsvBVqD5qtAFeE/suvIzFLAUTGdtJkHBPjWPD4oJQTY3vgmjB+2zdWKRe4i6B2byB/MjBFF40mfBiwRpd4HYSNDYltMy6rqcInnol7K4V2T5c9MtDasRupVtFYdvNu/Wxy9puVOO8oyLwK5AiEfVLDKBHmnAlgnz9/f9g164Aqx73GaHPdg8sp2lpMMwk29qsDgkfZqOBsRQFac+osl6qI/vWo3EwE3R/7i5BVp6xpesIuBA+eICIe2BUmrXAx15Gg3rgT5+S3DxEop4+VSSUylkctEEGdwIWDXRPKWamfBYPQ8DCNC7yRiAYMAq0YNJgIKRhCcpIGY9SN17g2KSfuzv5/oc38GQmomCrk5KVBQ7eoJwtM51vGmTeblpsiSZKopT3OLLcRCxis1mH2IKDJ7Yj4bliKXYjdyUb9Bm2x7DIRwEDV8YmGK1gwBXDU49riTGyvoyliMCViNiOXd9GkwDdIfOBU4bGd0e/U+MSTUgffT/7hskeoF/4jd1ZKhjH0E+bMKPsjaq6P2zswaQx2jD9mmVPm2oYSN5bjasSHLA5Adrr4ai5E/uv//BP/+d//56tbzcKkHIMxk/+T75iLn3SBEdx9eEdRVPGRZpgSGNw/AtC8D5dcJ+ezOkKVIhCtsjlRGp4MUm5E6WhcOiDT4BpRhZ87UM4mQKIjfJHS1fcisieJwvPdAa/AbltsblwZ/Nk8Py4o8YD4e5bGVazozfMxLhUgLtuovJChOxOI74QEM9GoNMFAQChC0XCKQSHYAHMM4YgGDsrgUDf6YArp4t7HLmhAnsLUQW/nFNaURKI/MpxlyxO7jwxaMhB9Nfy7yZcnTluHHr8rj/1xOqMg4L5bVKH/kRgNnG2LTQGumJ3etdW0Z1uiRDajhuJSQKa3p8EXrrwz2Y87Hc70NGYT25mUQAWqP+k2+sedV/WwR5TCNPGKD6N+91j+BAABVH/ySk/dTg/m0Kn7SkE5t5dP76DmGDRTt3GeQ2o4rjpq9j9IvrHz8NV4/z//uN//8+vDqHBObY6/5SOMavh5BjZHQtTL+asDuZYQ1RYiZfiVIjG+Tvhi4hHKAmYOH24AAP1VovYq8PxueqNfpWwx9EE5HlqHMqkxYZY/RDj38G6yuZGkc1gs/8iGYDKJiamPhb+gPHaGAcVvmsJjCaDxhp+Nw2N/K3rJHNgS+ebs22JkCzodyrsqWPNNk8bYHK84BYonnpeDMZP+GdsFbVjGI/LvTYkkZMb0LZGBbj8LP8IKCXRr3C2LM5oTNsktP0I0T+DIBKSuHYShP1j5PMrzkAtp9n4IV0D6zRogP/k/k2jwmKjzNuX45fj6fQsl53uEcK8GEduJD20n5CthfQZUhnQ6a9/8/evDnnOZRXdBmkSpqC9wEk0kC0wCkKnOCGIgMcXYweED5Krmuj+QeF8nWnHQFw3hVHjEOo+ZeeQ6IH/oLgGbZ0EBlYLUMP7zMQXEwQA5AEyoO+QrRXsDiYDkPvwKJGhv0ERjW00CR1j5kbB0fOOkeUL9Dg0pwUQee/b2FKjL0g0B0MA64sbSh5S4giPzeGXcFRDsy+hitzAH8GXWaJZUpqS3W+2EAsI4/UYFO7WIjw24CVBaFVyF8XbA8X6MApmYORjCRIEetJii3gGyT7kzQ7QdvAdB6RaTERREJGn0XIxSTDt5ysTWi5g+F1MtVHpSd8IFnuKjr+pAxdwKANWMAmsDd3INwlmUyY0OGTqQ7MLr/G6SRJGT89Zx+ocZQkc4pdLBQECzcDE/ok4OX557Bj4JWEum5nGk+701Hk+phf558aTk/HJeNpRdi9Z4dDQ/f/X/8TW9P2mAuqgpPJf/8tfs++BiwFq3Vp4faszxai91Ev5kzVQmSL0YvNn2Ck+nOLEjc/+cg2EKQJDOrkxDf43OfZGs3mw26UWbWCv0+v1xtqCdiGfjAMPJPlJj/eco+dnFRPaAxNaNW8hd3Byi3wmIyNb6/HOKv5nn5Usufiq8wbDPBHtsUhu0VQr+zkOkiRY9MlVbkGGL/zzNTBx8+qQrnOnmJvKXifz3Gv4s2mcr0GiN9/krnAfxsopbccNL7ovu06FjIDlWQCx6RQcSH/uOo7w99NDej+Jz1nWFfjCQleE9Bl4Kz92KZyhj5h1HG9xbN1oZIKI4pvrQ0E22jTP1cdglkdtcvhAffO044hZS6lH68np+AV4HH3fLCCkqdr5hhGqlFERYjxcLFnXOomZBI4uxvUhhjtrANVLYUfZ2b0icpz/2xtxR542ZghqjdOj60LHkElLCnRYZ5MEta/ahFhngxJBQCsecEbRUSLAaoGEtBjCaDFPTOGaHDeYzbkQ+o9N/IVXYgGMtcdgL0Hsw3zKtOztpiDBcv5UWt8WTS7b8nKWug73J4I60lE83EAKnIiWzLcr8KS1jxBmTJOUCBw0P+GQdet5WgQokz478LNrZbzJoMJI0ZiQIUGTSkPTTwqBOndBVGaR9RYFxzQoFoW8Tc78eiomhdxhGmGcDSECRarz4DMlmjS1EzetPKHMBp/3gXdgcuWcu4V/0JHgDO/Tp0foCLrKg/gzaJZPhpvN2uH0c1o9QQsXVLDJ+g0xDACIAM0CG+QmNhHB3GZ2HsDIngbyW5lL0iMdNCeUA04hbQNnhVOvmMBdypTUl/kdsA4eyrTNAH1UNxvpfTplL3lAT7WzBgkFd1sIRuJkCJ2NlC9Wd+D8vYAn0LgQeeG7hWyZw7wNohuzAC6J7sqBDn6F+I4UmaLU/5O1pD6/L3VW1CKtOJRJiFVTa0+mMvfCw5ScWES6BT/NXRpWUKr7sdyjdPd+rBRxIBmk1ZL0IlNXaZ1RfO4FR5N+A6QQ0a7JXg3AALeywG6AF/eDCXwb5RkyoIEK9p0+cMOyQVETzLdt2zRUC5BgpyBWYjURYcLe0h8wtri2JcoypBfCIIYTKDSll9lTK4xgHDbAM5tbIohagiIYJ5EplE6Avxvky23WNV2ZKoVCwQZEOQifP7iOUpU1QMBNwb/q4kYIjNVNMG9p6KAHaLaaEH0eS12eB7c+vKYYmJ7cziGoRiBubENetxRmObUAkpFF0KRqZsEqwSqT5Qn7KFfNMFEJgN5gV8eRYHcc48h8VRTCa0HrnNhCzi27XyA89MD+yvnYMrkU1jjo0os7V3gOM0tZBNJWIzvCsT/bJsYRe1YnQKbx9GlxAE+fsq9/9XeIfZgKB9w1xDwezlbA9QSYwvFOTjO9XaFA8MjaMednXIsIlzbBXhZG+fWv/hvIXhG72o/LOU1mEZX5w9SmuD6EqtavIRNJRoUYpb4f1E9uV1WS2LFOMp+VyXXe/y/s2zBk5iaztQx+AQ+ZI8h0gdRJWfeDvPdsOiAaGpA+I1y4wuwRL8nUYApJolLGo5tjQLkjqV1LB1ljnkzmpspKySAr05vZKhjipbo2m3moQot1+Nk9QcmSRy4oCcYjsbE3YICsHHDwAnw5HOEccDaH4LbkNILw04WKCbHrgpprdE0XslVP+KpBi+R8gGtetBwH5OluDtd5Az2HLAMPMl5TmsyPXR9sEjg0c4riE6mJgKnlQ9S72zNHBY+M8IiogzqXR3Qe0G/ZsiId9Gw38BgnK0blFkCjwnQ45LnZ3N+Y6i4gQsG75oZ9/Zu/rXkZDRsAtTEqjf4BLqPcL+bhO/rtQwZdmZ2X3C3N0QczLYWLO5sWw2MznyZ/xz0wqRQgyoXyGGIMHjm4hICTLsAd4LpYQpOgEu4285lxnJUmecpkybl/wVrWfFg//PYaixLU6jXGTGA8Y6VBJQ9TnHR3Yzm9VSYdZs6uX7D97gLl3IcIxTT3rc7jPBIM1ZZDtUJ/hsGqfEYhHz2qncp72CQgDCyfv9NLWYtZJVhAOmrem/C6xbZY7zSH/e7JyIKo2oPowTRsxJRlC/9KEAjUsP+8M1LMDwCoZL8plkkfDcgn4YlJ8oYnXJERSEpsdP6cLDzYQSvNQQoTuH9n3k/EjF/3BnW5o34QX1XtAyZiSIMhUMhyIdhYjQ5qZO/RCz66YGqeJq6XJyV0a02C8K4MS98AKsRym2ZEbbu51QeGoxJDcFGYfKk8qmgv1SssVrEg8HZMlIRt5MtVLdm8ZcVcFVwBj+OiwG3PpO+oEJGP0I/Cfbd3dNzU9mmOoVR0p63TF5TFx8ofzTrXy6A2kaAWJ5lagDcL0ghyLd1rmXaScWFcYpyO6YNoMs8eLheYGFNDa+lGSco9ewEROI4na4TGiD6zJqnDLXhvo4NGSpmUCBVeYpi9xNo1SLJM5RdxXqBTcCZ1Dr48HbpWJEIHHbljYQorbG7QP1xdfAQHslxYaSwccOc439DpW93pBm62Y1MABW1l4ULWGKdn332LK/oLKxQRrjZuvqEykXeXv2VpzB3er4dlzobdEWuz2bAzalZ7X9PLai+lKoocVLZQS6UZEz6ZCxgXxh7CorvM/9b63tXWuizlWSsUSvv9h0/XNKeRGH2KloxZmOrLZcQX+rpwOclbgA0O6XqTFVNR/qxlbVu8UECWKCSYmoW38AOp8LZcokLn5ZcUBhk+BDkub8cLl7XbP6ciumsDsgMEDSkTjteC+5aUSWK6viamtox9RpWGgjEhJPoEJgwgj7KciN9Cb1Sykwwm8bLlB3PIREUEF6nvJrGxNwGP58LzSEWx1DJE6MqOqIc4t6Av3YWAd4OjJqSxDlxlgHEBZSgnFVbSR6yo4g7aRW5oNuX8lglRkdEEcdO3LUOZfSI8hIrQR9iSVOfQjso2IQfkUhhxjokPe/gYzAsfHo3w9zgHgfJgywUUEHD1VbfTOyaxPlzjx9kTKdPGPUa2ANKAUBm1Sn5yr7VZQt5O3mySmCgGOD4ANljiDCz8yfQVCA+vFSi4tNVzTLWQ6YMkzAtJbuyWpC2C17U5ObqkL8Mbe6RjmFVukrCmIns/7LdPO6V6QwTYysYLVwvkhA5cAxClQKXthdg14lMshwa1X7rLgJmnHRY3LXYVfA7YJKUUmnszLJ+OZVGH5y4AaVCfOb/DBTOehbAl9apHBsiZ6e/BA2J5XUlZWuAoVUw0zj/KkUF4wXApCO7A2lNcX1wX2JHKfQ7GsXzOE7RZPjDZwIfoaA0+SbBwDSdjMW7wbRwv3qkize8/fCwKyOkLJSGnPSkipx0pGqcnShBeHqlpXkgYZHmnaaBAKqvYytcS4fIbDE5M40fwMoa2ldhCLqPoFgfVaE01163VkmUO77Xsb6L6Ex3eOershHaN9lfbYWx/PB0fOad4+Yd/fa3V/3ct9h5G8xwXcU+lscUIAyRgMod0qpwnY7qUlFMdjNFapBpA4RZLKWwmGuWygBlvJv25GV9SCfWcQuD5sA0W43yAjCm1mlSXdekzigegw2JCgS/KGQWhq7URES0lIOgk51hk0C1/FdLKgKFr+MCa/Q7sIOb8L0/JmLXW78F3v4eHuPy9IkvZJNsmffheX6KmGlblqQYY/mmnP6qUSxAj8mz4VRh4d7iWxkJALIkHjTXgumngHIk3aPiQYWEhTRTcgKqtsapkDyZTQ7Zs0/ohtO9ZJ2Xydq3nm8bheSHvI4ko4IOLrFq19fLwUbhip1mxTmlFubCKubPib91QvD46apTRaai16sYmW77dbC9o7wY82QWucb5G2diwV+PzNckhOah1upFrwrSUrKiAVgTTCTAzMpLXliWLTKETanTOOrlYlQSRwlKPbyhgw28V6F+TuNuEJSR3EFY33WqoFvU3dWVt0Md1xMf8M4SiSMmMRg1li2UROtIUg1kCi+2IFtuUBmdlk6xjPPGKrlbdQaPTYKseSOvvQLrv4H6NatixunjbK91qRVBjOTnRj9oOj+c8ivjdoHHMjlHEqUcM3XBJoKaWTlLj/Os//BP7RKWbTlAt9SUW49SogWMxKJ+R5vKn0pQzK3NW9SBNPfTw979n1wF4blmmq0nyC6s8ygyVWFVZXy0eqBExLYUvsQCkpwpAqPaM/drlH1TCeRvxsI8/VLtJNSBrZNqmthJkR8yBPhkSPxXdUCWOfFJNBcEEHG5VVOxMArGgJ7yVt7/T5Sf15STxckZ1vN8GK5Bn1mEgzGz9HkQW9+iIaCku4lBMkitMW3KzvqM+8kWnplZHVssgiVgHiKR1abNuNKS0kmdBZQRkHs4jRfYKN6ADDZZsWHNTKSk9OHj9CQukMdocB85di1mysJSYzQGxiK23q2nYv5EZIoe0+qDmC0hJ2ooeJx0sSsrbn7GSGG4OnsydiM27bF0oBEIJZopSHdUkZOtyVa9u0ENSMmwWu46ooDvuOr3jM7azuopVlAlpyIr1VAg24bNfBS6ia429AJfsVS/yttyZtB0lGiroiiCyzxJTnsyw7K0A5Z4qoheTY2c6bdYxajqdFh8f6GApq3B9UWHwtn2qvF61wco7wS2WB7HeMbxXeJycAOaRO4shJbJQgNlaCfaG6eceH8PvutxJryxkm4NpECQoiVpJUFVLDVDi1RpY6nqOnbo6gduR1FyqpZ1QrVmLBc1yRRZdxta3PBZmGLlY12nPUzEwXN9xZ1hc4os0iSARpqexB1Hl/tkHpO1gmIN+FwQzT3wHT025XxdzDiMz7OjpYu7HbQzup8aoacXg27MOUK/tXBJsCkkNpcnwbd172+HRTaFRDgzFswaalFGEVtdAg9OtquCIlzbJG7Yicc5hFd5mgFST8iBxUiaDIu2LHl/+SoPQ7yu4kHiVIUl7k+GTuIkn6vsqzG2HabKP6LUNdlM9TZLAt2MBNtZB+aphwKQ3PSIkq233Y3of6IziD4C/k7STuZjcgN4r6tZgr1m+v+VO9lc+qxeEnY0qeEt9HwsqG1N7pmWliZvMUeO/RTmITRKEgXFxKWtGcB8HrjPjajvq7YB+ods4HoCXrdX4z/HAMJuD87UTTCDh9BMLpdWaeDyOsXrZAo9hNhA93LTYxIkcRywKW7s0SlfBrSk8sbBdZ2CAuzQqC5DQ5CNAAaOLafMTVkX6J39rQv6ytD5c6uhnnNZTm5uoEKdZraJpP/I/3BgB2Lhffk5dETlBCZzu+zVtDzLjCcTJgyMqcbdljnzUwwoLPX50rFUCFOFc83FpenBHG9NQO3SMHW2LpYfwyQcs1zVJuAay4iBbkkaZuAtBWELXM1oPXiVkVAFMq6MD4+rdtxdGtqesd9Rp7oSjB3ExmYDxdAPfND7+8Z9jVWrJTAFBsOdw3G4cqeIPrMZsAnhchKvl6a4+UCDuaUrqBJpcQ6a3EpMyffYMu9vbM+ysUEFM61hySYNFGbP+vP1RoeXuDsF7/Rk7lCXB2529L5XTVvp7FHBbH6EAXbyJgpCMxr1EGMqS2xbVnlIZKpEFgxP5Qpaj0hts8xA1UF/iNzmcrAN8RDdV+MbofthL7mE0tgtnRdQfgBK+2iOLGmI8SueyzXzMvHJnmXZVCkh2SJSk/xWmVibW8plYRYSFQxLxC7mRFQsLpT9Tu4IebltYbjBlaP24wV14tIs9ElR9+tCBSUGUY/sECZOIzI7Va7GTFutaL2TN/qBjdboZCxYiibLasYcaO9yBjWzTm7AfYdcKJb+E5WsVNZhytU9hpTZv0/CpFmnHZu17eqkQAkbdwwL1jvU8I0VOCQz9ffBt3HkcNV4DivjRI4hA0x3Jlgiq82kyKcQYKUPvEmhB9RTGo4RQVYIMZKClD2AQuFQhjx5g5ok62uYZ+w4Co/mPbz42LWY8shNWOJKmz07UcTkQRJCk4KZMIW88cNjBPewDJlfs49DA9VM0SXgCDv7FI26wOlOSChfvFfmGSLbR8GjUfCyhFKX1mRpmuGreg+gjvXS2L2Cf7P+71E14hFmqcx+dFK1w/XwfRD0guQ36kXZInhPxx//psztZVfkYQacPKqykw5wyQ5sf6KSQLXSIAbTs8h5x0ZsS9hHhR0IFN4wy3O26G+I4UabpW0rFsjC1fiO5UQiMZ8JHn+F+gXF5sz1dkHjvL3EqHtxEBU6yYK057L8c7YSL+yVX+zmCBcTynKfYxN1ZlCnHgyF56lFGrbefBbTZ6RhKYfxHN1Zl/fG+UH5mb1H2YoIVDO4X4r+kW7zYQ7cZ9ySAd2hPojsdnF0HE85SP6ATWnBHgYuF96UdBY+xmvKchHjQyyK7E0yAaD+8LCkUt0r8dwKdY0GLRPY7SAGoIBrS4dRPwOfiAUDAgVzo/8OHy0IB7kOIfrHkPp6UtY/ij7BOtJWm5C6PaHW3xcpWtTNSnrP7JxrXS3I15tf/+C9d3PUlj/W5R7/l5qgSdl0sV3h+vIVfV+N31MujTPp8Tx+06aoMXh+jJv+W++jpPvCwtceQQeHzGs+4YYs//jP6yoeEXg/hn9xOCI1/ky7GOILe0XFGgU9igSeZGbjxQExcPA5tcE+qkm1ZrNhUtTleG2tP8CSI3H3xmN67ViYwnlUHcSmGYyoSO8mgfgdNKKZiXO+23ANf7morQW93XwJc+tHwsaxFgX/nojWPXbZw0VokEebu5h/+tfnnYUW2ra4awhwjPi/QknQ7hN/xcScLYC6HRvbhQxKsPfJ1JWIwYNKFPsR9qs19JQoeq6MJAS312tBSn4uVinzVBq3HxRXfB7hxtrh/5VHTI2Nbny5xj4Glx5irg48zaF7AyL3cd4V9M7t7enC2mE266NjgPsi/KDy6F/qWu72kulGQfQ+If+/ncq9QcWbzvi8e6/H2KfU29j+KiKUxzfrhuvUhk0VoDwgbohyzHWP5E2ZURTwRwBn3AXOpL4qTp7gOWee0Fzojfn/98XuzXPq4PTTckVVoXt6UhoV/b2Os28VSyKmaNC8cTLQND0OaArzifsma1njUZhpXKGrUcKBG8SkOjx+i6w+0uUs6glQiQ8DNbVXJ53Vf7vF7SxsygxpA73TSsB8MCNmWVVIQ3qj9oOBnQPApjUiw2OUZyw4ajMFmSCv4INtfq11ql7De6c7D0Luz5WSDrAKsbHQHJoCFBNkAjHWMg+3KGYauj9etMddOCl3BuKlaEoWmpdjRksRs5URpKamRW9pHB+W5EGsy5/5MmEWUW0z/HdbvkofUt7B3DwyGNfHcyY2pz7aoo+TwVznv4tc59eIXn31RG0cQu56xIdqMERav+pM0Ahc+ubOpThpDEdxKLiLy+cX3WZJ7UDe3TcmR2tdqyFNfCiYBTY+a9b7GS7NrdSpvwTwB94qmr6WOjFjgoOp7aDHwtqmo5mOUbSp5kPv7+8ws7IRs5duOmrqXIeSXLZm5FeQKHloxbZkz8810+hMkZkEEC53S3l+AOdbiT4FES3v9Eb2Cz1vS1e6gdGFvTwF2tntJo0Eerlh0vwjkpq8/fegEg0hrgi3hYGLs+0RlwVc2uePnx6WtVwgKj996vJcteVwPM97owP709urHt1ef5L43fUg16KSN1Scm/uR7BV67IoIg3wtQSnAugLOJ8BxOwALG/QTM9R0bpzHNIOA+/ijB3f1jdXp6cR9fMLkRSb4pQm4gdX2s/cEZyVsxLjp2zJw0rlYYhOZNzV5Qp3i4Vb9ySsTdttNzQAKCWFTCl907Sba27SVVrIwk9X2Biww5crgjrR6xxErkOQWJxgFJESIpIjLgSKkWI3o9Y9nBnJk7lzS05B+TVtzjBww7tsYu7o01Ohb9DzPZmrhEyVtYR5wfPtHG+W3Ypc3DGgLgr0TL46k/mduSzTS8wYuXz/XAaKCDLfnLNAje5QVQuV5JoCbVPEY2utxBYWjqMfUl6Sl7VzuisIZX7FqPwukp4Se2PCTExlKK7DMwn/KYBPlAr1p262YxaLJLODbVSw6G1R2XxelJZTC0VA21SuDhJaT626aAdHaS/TsFTlYcthKlU+oLTYw9W3Lhs+L5QOUdcmCiZoIWSts/0+8POGe72rOWZ8yTJIz7h4czENl0bE2CRQGTIlKH4GgFj0V8iGVncXKIkTAa3sPd6ydG4fs2ECpdtfnCeX5MzhanWSr7p2GskznYbhxli3WCFycnJesKj8uiCnYI/z2DGCwzeBEYbAoCNTA47oXR8tstnzWhm5f3+eITnKiW//6DlQQ34P7SyIv5VJinzf+fClCUecbTZD4wcVStDM+axR5sZi9ADCDjqq0jeriu/CL1kI6RAnaZrPY6ncXYqGoNuo+SygCli8d9lgyjVJSyChWP65iUd4pe0ubvIUlQbvCNdtsP2nhmszxmRT6Cbh+Sg0xJS0BJur0XxLhuf4282jxk9kru5BwUMfxw+RZjbIi6o+LzT9dvgKqFHaGFmLFyaljmaOSJReVTN9kr9rxTOXBDbrLAntTWUgvlwCvvqabcHE9qExAFYrZmRpl9GPL2l077tD169pMF7Mm58RPaDJzscOUJhNXcGNRuse2JJMMXFpbkhWZNhglBCb8pHLBTfw6UjvmGdM4d+mcb/XNhnKPyMVGl86HKQqlkZaTo9JD9j3SmlWnk/wpNZZcjWt8szMAhb52/pyD8Nqb5C1zD8fHMCyb/HRyaAZL/ko78V3IiWW8VWJZlVGLgarBUb5G2rBIZG2kJ6m3NbtNRK/qPd6a5OBBihBGa3hoWbdkN5RegtTT9OeIH/w+gFSDo", "estudio.html": "eNq1W1tv20iWfvevYJTuFplQ1M1yYsl0NrGdy8CZBHF6egDDSIpkSWJMsjQkJctWC5hfsC8L7O7boh92gX1YYIF9mLd9mPyT/gX7E/Y7VbzpYifdPRMgEVl16pxT536qmIN7nnDT6wnXxmkYHO4c0I8WsGhk13hSOzwYc+YdHoQ8ZZo7ZnHCU7s2TYeNxzUAy+GIhdyuzXx+NRFxWtNcEaU8AtiV76Vj2+Mz3+UN+WL6kZ/6LGgkLgu43SYcqZ8G/PAkSaeeL7Tu8UFTjewcJOk1/fZjIdJFo+GM+vfbnXa3/XjQaExYxAO8O22vs4v3wI94/36HdbzuHl7Tedq/zx/zfc7xFk7xts/2Pcbwxly3f7/n9JxhC2/iEliG+96es9x5sHDEvJH4N3406jsi9njcwMiSZGI6wrtehCwe+VG/NRhzfzRO++1W69uBw9zLUSymkdefsVgnTo2BKwIRZ+/gxhgMIZZ+e3cy15LrJOVhY+qbDTaZBLyhBswzPhJc+/6VmbAoaSQ89ocDMePxMBBX/bHveTxa7txPxWQxEQnkKKI+cxIRTFM+wCiYmcwHAR+m6inOOMSj5yeTgF33hwGfD0Zs0n+MQRb4o6jhg3LSd6EwHg9uGn7k8Xm/NyDIxlUMUPpnuWMlfLRYQbOxa6kSY6Dk1m/TRkXge5qaJQXlk42Yef40UbxNmOeRvLsTxVpnMlfkNGeapiIyLSeNFhVqaQzxTFgMllekDC0X1FtrlB5VCOFZa4OK0ogfjSFoYJrGCVBNhC8lcTWGYBog4/J+JEoZZExZYoWl+x23M+yyDaWvrOlDfMwJuLcQQOun132r28vpenzIpkFKCyYLkm+/jWfPK1Ud84Cl/ozL4cN1oXy9Crbw6HmaFfJoWig4EhEfbBqZsqiWMrb2Nsv/jTawh+fQj1Sw6Lf3aS5k80bmbXut2bj0CDZNhWTeEhMere3ACYR7iVka1JipfpUeVmEGGS1y5JTP04Z0iz650aYPSwCPuyJmUjJKThnzj8muiOPVDdKeKnKSSworvcMEC+b7Y9ryyhbU0BYLzBclIQuCxYZ3QG4s7Us1IpLw+eQWOxKXhZqqOGiYOG5c5fpoAY0jwFK4JSapib70NRmWeiRj8t6hiEPlx7Bp/ke9gRmjEn1WAo0MRZ4fc1eiB0PTMNoWvfLARhajtLq/N7sChyzyw2SxEQRJM6txbvBpmqT+8LqR5bAMc44iN6Df7HeVUJWbSWfND74YotaYsrbxhVS3VZFyXA3cHw6HQOVHQ7FFhSqbIGlJn98jHjeNiiwCSZMrTeda3B3QvnPXbVs9UAn4iEfe4rbIItdnVvNozfd3e+u+X/WqrBBw3V8degrvXdvOSkgs9lbsRfP82XbL2mKgWfUAtFprmSPwF3kEwqKirNgII90KL34kRZvFuPtw6m0y9SMUa4gxK9xt4Wq7zW/qOd98d7njsmjGkvVge9DMKraDxI39SapRXWnX/JDqwpBNaoeL7CWp9Re1dBxzXuvXxmk6SfrNputF1qfE44E/i62Ip81oEjYl0D+0rPZey2o1nakfeGrMCoU3DThW1EyFqglNiihp/jKUfM5C1GBJ81MSNmvLJTYhmccumqr2pcIPb1C05nt2DY6AslXT5IAbsCSxa0jyNTlJVXAipwGgHFPzWMoaMyzk81q+QES1w/dIJtOYHTQV3OE6PEocCOrwrc9vWFJAbUWt4jEB80Qkt2IE8eva4dPY9YOAVTEeNLGZjU1hn1snPE9tNiCitYJKNouEUjv8+V//rJ1Ob7Sf//l/S24qKChJbRNTQF2H7ARqeUuwfTcEOIeZ+iIGZPak6TB94w5RYRV+4FYux7Kn+eOtJLyYhcj0LqsdHuPx80/0fCs0fCEC5Fv60fQYfU/CYi1Vai7ZUjK9U7IuC8Ptgv2/f/vHf9eOPv+EUMJ+sXRdu9YyW7XD51Q78+37AEz7sYQ6QfUbeOxWsH0Jdco8ofk3f7LuEDuAGxVoRDXrVqzdntlGb9lt7t4K0jIft8iMY9+5nbshKvfa4RGrOtiXBY+SaLvcT+YUt6DOO2We41AKWKGX/+RhhIK25G/EIsgE0YwH4uc//8cGnMwShScW45SwN0dV8izjE43JKuEWVy4CVypEsB64aDwW6UrQeorE28Ag23Qaabgp/OoI1hVXADaQXqGYg5+I4PN/jUS0JWARUHLJAzLCP015wFOxHWYM9uAT//QX7YhNVqJpGdXyn5WkpLIHNqwSkvZAY4n2/uW7kxNtGItQq8ssUR/k84s3seOnR0iRMQS1rALlecfNJpsroMg/FSwvTt8/P4XmUb5tQxHIqaRZgq0uP3739OjNV6yvwK0ieCdEeBLN/FhEsNF0KxJezifNtQUK2w62mqTaN3ZiH3rCncoZ6Cm+PoOu3FTEemIMMqjnr05PzuwFwmD/vK7M/MMVd6xR4NTNfIBsFKFhdVBFTv8mn7gwZUoEmonMidlglvxomLJfNkq5DkNMJTs1uMx5+v2roxN7sc5Nv/7i9JmGrMZjoWFQ0xH0XaE91H739uSFsZ1btSgb0XQsM7UXwhOpqb2XdcqnpLISda41dOZY9fzZH+mkTOO5gWv695GfXpva91HMWWBqr32UJgKLd7RsueUEiAVY/Ix+OfKdGjFul5liL/ErhEpgpFrKbsISzidAvnn2O0ClMsuJCtkPsJ6YJyguu56VpAFAz96fatno5/+JtO6xpodhhY1AeO0PvVbGwumb41ysaPRWoDof2r0KVDi9ziHbvW8lDxVd9+uqGNJgzNp4Cm0DV6n0fv01mzCkF02OYa6qfTCNilaj1pNhauZ7XHxAovUjFlvhZBcAf6AxpUuX9lWAjWD/XZSLVTC8//W/JQSk+GFC6Z5Zk2gEgFchQ9AmRuSkpibry8JzYqk+HtsRv1JRx/qBOy9O32Xj+oJFdE7qswRt8pSbJGkezzgM8grdyrPpcIgmh6aWcLQcnYWC/60/58E7Op7QX7N0bGF/ujqCLWfMjmEMqovO0O7ofhTx+IfsmBaPL6VfVdGLaTqZIrKhMTijkzFbsX727sWzcrBEjDDMoZEJGM4gnx6dnD33g9B335dzFU7GMNsrDFs8kidlNu1wsFPEGCrFLazCgqMxWgG9WOohTAWcgIrAk6CR4RUJn9G7bgzkuFU2jxUQuQu9NVfHzAWmSRjzsAL29vW7k9cvgCVmFOtyJqilJnAEUNT2tlxlUYBVlGn9WjjVDRM9yK5hKbflOUGYZZUcQvqETj9m/IiFIKqjSCq11axoi9C1zU6rVfDupoFEtZKUdBAwt8oO4Jnwj9EXkeakCmiYeu53yPwp3zZ2NuHQV9vq5IRHsV8V7Qu8vuTBBMbdbpm7qAbnHd5tq4dOZ7fj5ppBGtJpcbEFdPyi6iqveTLWK+JBvc1fcBHyNL7WOy1s3zAras+MiloEFujVw9elIY/ERGxRRUPdszW3G9Jt3r5qdvI5tAgQvcKjtl4yKkEKTsc89CuMvsRrMhlDyKekHBjWUP7Blrv4s7trWnvVXdPy0nynUQXVcX7+xYI1XB1gmEaWy5J0hUWMKX+y0HqTg5OjQzy7j036p0oXsAXZ2A/vJvv4MWMg264iwCIg2EF20dwpVDUNAjNEuMkemUSR2Isl0rJwL6seR+9wA0fMK6PPxLyLwZewp8cmFYLKchQ2qnwJFZsxX9n2GTZWbEDVQHK8LKP03Dc9yuhyslIjYVaOk4SOuYvkFL+FEej1X3uOEPhO0pQom6MgHTbrxkCxJSlUCEsYEt1wGkkhabKp1i+NBQpYxfKpvch64v55qXXzvGPumt0Ls9QIrMlsWy2T9H9h5u1xtqjjeWbXwrKeCQOgZc7Qe4xlVs+09mmpWkbFetEjq7WstQenAsR5YxeE5eK91qMW0QRGOJL8q9YXLXOF2S5onHfNntlo09rhsNd79Mgk3Vpgut0zhyxIsFY20JV11q553sKynlqlXAfr9iRdtWh5fnkxANPSByh8k4hf8rl+et66UK5Bx6VRAp+3T8/bF3IoPy2TPmFZ1ul558KwwmmQ+pPg+sxlAYv1l0iSfriOtXuhhqtYdyULNLqCuNF58NKkv43uAyAj915Z1rvIfKhScmP40cUTlUL6ZO+bIK8qKPYkZX+o36N1xmILjYfW3nJnWVqYA1/TjQUc1ZpO0C4jD6exP/9BxEiopEQ6J50jalzyk3CSXhe+42x4KNEmPGj3ZkhSXBf24QLMCMtPKEob+cvZJSUpT43Bzy5lJRplDACRgJhDFBf8GaVkKnGIyYGD4ck1UDjlsEEVQHCtuN7FXFjybyxls8lhGSAzypLCF3AXYF9NhOQzjSBM3TGWS6qlkL0hWTGvyHkIT+A6u7HhGzyw2x3p00r6mURdmzCNeHokz1wrSe0Pso2CiA0zyYFklbYNZPDSVrUem+uJdW2iBlCqUSGvOjk3CSCxbowHbWu/KWdSFlFBYA3F7EGe/FDdGsiABRJms5tikg6IuM2D6oAEZWsO4FrzhxIGDYjODAXvikTnxgPPdK3rcjYbuXlYwKzDqy2h6EB0gjSU9ugeA1QjzmLba6Lwka9DenvQUy/Kxt7G4pNKZUqnuqp1CgOsegi1RQlchPJZarfMmd0a3G3mWcljl8Y0mD20RxZLQcuB6SWlYFxYWTpIaVoeqD/JftVEs9u/Y1mzS+ZWaOUrTAOw3+h1eb9TNyxVKr5/fWp//GYhpStLYD1FCSpOBX2VcQbS0Ug3lhoePv8UjaaBSLS//kX7ZjHbAjT7/J8xwj1PDpz48JsFLAxAz5H7Pb2D6c//otHg9bbBm5XB8OND8l2dOLVkc5c8+ajoVoaWqu9LPvbrdTg7S64jVysUx+fotxJ9aCyggUwpsc2umI8qkqfuWB+aC6hnLLx+/eXJ0+N66bxoby6XLiMgbiyyQZljlsuddULJWFzpdNhfydMui7zElscd5zR1MSD7GfoBl3XLAJ2nnhW0mhgqcANWJOsYa8yIb2Mh4YcDJ+bscpmFdhozFEe5PqmcgD7lhYuV3cPYdbriqZfmMcq2ntUe9POU9qETQniwNGIOLaQsMCpYqRs5yr7i+bh5RKmtmE671XrAJW7uNXNky28/KjsFAXiOsVBJDA2RmHE5ssRMWdmtAVQmBmvl31KlHDiPXPEFvxQbdbHYVs8LtGlT1FjhESigjZFqH8hUAnSkTNu263SkVDdkKpC9xFpLcoY46qFwKlsNdYWGSmvfcd09E/IajSOeJH2r1zPp26lAvbV6Slgr1LJjrTpt4qspQgxQnmxjs5ODCtFHSyQrRaks3UkXg/W8FEYysIR+ZF1LCRdR6Lphh1HWGJWDiI/Usa2OAGnZAwzKHmBAliYPpVcCUr2e2QvCH53ASHAr4NEohSIzTMX2n+Ygr2mi2EVu+fLq1D6vv+NgiU6JjtRBDz2JGE0vHs5YMPVY9pTKh2fwRPnwViT8w/v6hcK5wlEi4lTXmekY9qEu6ajw/QaebNG3cMbD/X3j2/39xtqsszKbsVsGBZeCQpWSschkdu7KlRe2FILlBv7kqZyh7Cex5Nt27OKoxEUASXnW1+t1dSpep6qn6t0KM1wC8cPlev0D9q5JKBGBDlo1Hduk4KIrUGNFe9WjGCervSRwznmugIsn+VO/ukWU6BlWtbUsG9MhhsRGARSB8QwhIPPJ29y9WmDmK7LjMWKYggj4zb/Asu9lMNVJectxigSSR6G6FBh5C09/QCOsEzDdXawA07E/86NEgRcJV93bbITode/+8cd8RJ5t15/U5WV6vV+nDw7qG9hKf1nH9KR+4BT3v4eUjY+YxzQZgzRbm0Yqc1o0cyxgk9iWFvIb7CQytYBp9MkQo0FPOAGz6n2pznX2iEh+Iy2p1B+qlK3OaX/8EXn5YV1TbybIssppbV85uaqM0RPLXhcqPVUnjVndNbg9wRVC2X7t8DQIsFRewmu5xaOsjU8YsrpjHzoVvaViNAqUkk3HomtDqNmaYa8yr8uKMDvQoGCjolhZI0ozj/LKD+VxZvHRxYBSNsvztUqAhOG777KHe7bNjOwZparH30zRhHaNAbPomDfVDTn6KqJBSxIyBjkb7Eu7r34U9LW7r4YE24424oFhVKvjzBtEJJtIua07UrD+NI7ZNV7lr14mM+NJ+dw/L58vSqZD+zC0yOOkxdigiPy1LA6apNnY9ex0pF45KN6US/7R5LpI8jhH1RCHuYkJOoUJG8kAVR4b2Y6lPjnNImo2LmxveyhAXFSx426WKtzM7cP5thgkMRnSqIRRpUbZO5umpF4QwvjJDA8ERGfUel3usG5SKP8bsWPcKetz9UXExW2yJkYWhf5K5wvWI8Lya+i4X6CzenSt0ghFmFikd0Z9n1J2zppLMZC+KUA99otb+Aw0tTcn0SGb6rRljtrq5QOr9Yi64LyYWW+oU4CnaOBTNMoA7vVUF1vtiNO1xlbFoWXRFJwzE6VEZWcWYquPEsGENEI20X8/DR0qvLPzC5N/SQ13BtyqLmTXVIm1ZEalJu7S2r3V9636q4a0VXBjqei4abxGJ89EO5XUvsqHcvPIvvcVyX+wFabKFyLYoBI/M76yAuQOwhslyhbCGzDrhJVNU9iu9lTlRUl2KJfd0siSugQswrPl8Uk6fo+gm/nSNhj6PlH9949O9Z6h0tYp+/57Zo5FaFX+q4Bq9EIru/6x5fWPzCbZceHXNqCDvzfTOYftweoG1FmE5Dm3m/Gm4xS3eupBl9syEUjyKMRu7RFYnUqQMbKtveVuEOZ0DMf9/t2pXvfpgrtJl920Am10ROUaOmT1EY68BseEZEsnK9/MSfTpwA1XSWlBcY7JO057+73m3Qd5X3uVvX5XXjSRp0JM9IrbeeiMqAynuH7M0RzqWYfy3XcrjYq3GWxvkz7R3tHlMZIkBLOjMyp5maVOafTqOVU99kcfZF39KZF9hfyFLIsTqjKio24vrsHeOCQca4belieqMGexO/ZnAqX5Yqn6k7WTqN/4WY5Z/USj+j3G6vcXFwb4VHem25jIGL/k14lOn+cYa+BKOFB+6CeIJsg755ZFvxcyaWXnc8obFWxxGGioEzaFKxPBr01nZQepDvoq2ezCSgSSCXhYOdGrHNeGNnlt9n1g/euksHFA+Au8eFh1Tnopu0dCfj68ePhRO5D/V+Twm8UwLwQsbHoChzDoY2g5+RGhqNrrMxk213iXbBK/9zLkv43Xak8y3EpfqVkWFXVAk0CXBiXz4htu9Kj08fZBU/3nxv8H1J+7UQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()